In [0]:
import pyspark.sql.functions as F

In [0]:
%sql
CREATE CATALOG retail_analytics

In [0]:
%sql
USE CATALOG retail_analytics

In [0]:
%sql
CREATE SCHEMA reatail_dev

In [0]:
df = spark.read.format('csv') \
               .option("header","True") \
               .option("inferSchema","True") \
               .load("/Volumes/uci_catalog/uci_schema/uci_volume/online_retail_II.csv")

In [0]:
df.display()

In [0]:
df.printSchema()

In [0]:
from pyspark.sql.functions import current_timestamp, col


bronze_df = (df.withColumn("data_ingestion_timestamp", current_timestamp())\
        .withColumn("source_file_name", col("_metadata.file_name"))
        .withColumnRenamed("Customer ID", "Customer_id")
)

In [0]:
%sql
DROP TABLE IF EXISTS retail_analytics.reatail_dev.bronze_retail_raw

In [0]:
bronze_table = "retail_analytics.reatail_dev.bronze_retail_raw"

(
    bronze_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(bronze_table)
)

In [0]:
%sql

SELECT * FROM retail_analytics.reatail_dev.bronze_retail_raw



In [0]:
%sql
SELECT COUNT(*) FROM retail_analytics.reatail_dev.bronze_retail_raw

In [0]:
%sql
select count(*) as customer_id_count from retail_analytics.reatail_dev.bronze_retail_raw where Customer_id is null



In [0]:
import pyspark.sql.functions as F

#Null quality checks
df.select(
    F.count(F.lit(1)).alias("total_no_of_rows"),
    F.sum(F.when(F.col("Customer ID").isNull(), 1).otherwise(0)).alias("null_customer_id_rows"),
    F.sum(F.when(F.col("Invoice").isNull(), 1).otherwise(0)).alias("null_invoice_rows"),
    F.sum(F.when(F.col("InvoiceDate").isNull(), 1).otherwise(0)).alias("null_invoicedate_rows"),
    F.sum(F.when(F.col("Price").isNull(), 1).otherwise(0)).alias("null_price_rows"),
    F.sum(F.when(F.col("Quantity").isNull(), 1).otherwise(0)).alias("null_quantity_rows"),
    F.sum(F.when(F.col("StockCode").isNull(), 1).otherwise(0)).alias("null_stockcode_rows"),
    F.sum(F.when(F.col("Quantity") < 0, 1).otherwise(0)).alias("negative quantity rows"),
    F.sum(F.when(F.col("Price") < 0, 1).otherwise(0)).alias("negative quantity rows"),
    F.sum(F.when(F.col("Price") == 0, 1).otherwise(0)).alias("Zero Price rows"),
    F.sum(F.when(F.col("Quantity") == 0, 1).otherwise(0)).alias("Zero quantity rows")).display()

In [0]:
import pyspark.sql.functions as F

# Duplicate check 1: fully duplicated rows (identical across ALL columns)
total_rows = df.count()
distinct_rows = df.dropDuplicates().count()
print(f"Total rows: {total_rows}")
print(f"Distinct rows: {distinct_rows}")
print(f"Fully duplicate rows: {total_rows - distinct_rows}")

In [0]:
import pyspark.sql.functions as F

(df.filter(F.col("Quantity") < 0).select(
    F.count(F.lit(1)).alias("total_negative_qty_rows"),
    F.sum(F.when(~F.col("Invoice").contains("C"), 1).otherwise(0)).alias("normal_invoice_rows"),
    F.sum(F.when(F.col("Invoice").contains("C"), 1).otherwise(0)).alias("cancelled_invoice_rows")
).display())

In [0]:
import pyspark.sql.functions as F

(df.filter(F.col("Quantity") == 0).select(
    F.count(F.lit(1)).alias("total_zero_qty_rows"),
    F.sum(F.when(~F.col("Invoice").contains("C"), 1).otherwise(0)).alias("normal_invoice_rows"),
    F.sum(F.when(F.col("Invoice").contains("C"), 1).otherwise(0)).alias("cancelled_invoice_rows")
).display())

In [0]:
(df.filter(F.col("price") == 0).select(
    F.count(F.lit(1)).alias("total_zero_price_rows"),
    F.sum(F.when(~F.col("Invoice").contains("C"), 1).otherwise(0)).alias("normal_invoice_rows"),
    F.sum(F.when(F.col("Invoice").contains("C"), 1).otherwise(0)).alias("cancelled_invoice_rows"),
    F.sum(F.when(F.col("Customer ID").isNull(),1).otherwise(0)).alias("Null_customer_id_counts"),
    F.sum(F.when(F.col("Quantity") < 0 ,1).otherwise(0)).alias("negative_quantity_counts")
).display())

In [0]:
df.filter((F.col("Price") == 0) & (F.col("Quantity") < 0)).groupBy()

source_columns = [
    "Invoice",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "Price",
    "Customer ID",
    "Country"
]

adjustment_df = df.filter(
    (F.col("Price") == 0) &
    (F.col("Quantity") < 0)
)

duplicate_groups = (
    adjustment_df
    .groupBy(*source_columns)
    .agg(F.count(F.lit(1)).alias("record_count"))
    .filter(F.col("record_count") > 1)
)

duplicate_groups.agg(
    F.sum(F.col("record_count") - 1).alias("extra_duplicate_rows"),
    F.count(F.lit(1)).alias("duplicate_groups")
).display()

df.filter((F.col("Price") < 0)).display()



(
    df.filter((F.col("Price") == 0) & (F.col("Quantity") < 0)).select(
    F.count(F.lit(1)).alias("total_invalid_price_rows"),
    F.sum(F.when(F.col("Description").isNull(),1).otherwise(0)).alias("Null description"))
)
    

df.filter(
    (F.col("Price") == 0) &
    (F.col("Quantity") < 0) &
    (F.col("Description").isNotNull())
).groupBy("Description").agg(
    F.count(F.lit(1)).alias("count_of_description")
).orderBy(
    F.desc("count_of_description")
)

In [0]:
df.filter(F.col("price") == 0).display()

In [0]:
df.filter(F.col("Price") < 0).display()